# Tratamento Inicial dos Dados Brutos de Qualidade da Água

## 1. Introdução

Este documento descreve os procedimentos metodológicos adotados para o tratamento inicial dos dados brutos de qualidade da água fornecidos pelo Instituto Estadual do Ambiente (INEA).

O objetivo desta etapa é transformar os dados originais, que apresentam inconsistências estruturais e redundâncias, em uma base consolidada, padronizada e tecnicamente adequada para análises estatísticas posteriores.

O período analisado compreende os anos de **2012 a 2025**, conforme disponibilidade na base original.

## 2. Conceito de Limite de Detecção (LD)

O Limite de Detecção (LD) representa o menor valor que pode ser mensurado com confiabilidade pelo método analítico empregado.

Quando um valor real encontra-se abaixo desse limite, o resultado pode ser reportado da seguinte forma:

- A coluna da variável apresenta o valor mínimo detectável.
- A coluna de LD apresenta um indicador, como "<".

Exemplo conceitual:

Se o equipamento mede pH apenas entre 6 e 10 e o valor real for 5.9:

- A coluna `pH` poderá registrar 6.
- A coluna `LD` associada apresentará "<".

Isso indica que o valor verdadeiro é inferior ao limite analítico.

Esse tipo de dado é classificado como **censurado à esquerda**, sendo relevante para análises estatísticas específicas.

### 2.1. Mudanças nas planilhas de 2024 e 2025

A partir de 2024 o INEA alterou a forma de reportar o LD:

- **2012–2023:** uma coluna `LD` adjacente (à esquerda) de cada variável, com valores `<`, `>` ou em branco.
- **2024:** mesma posição (à esquerda), porém renomeada para `<Variável> Status`. Continua usando `<`, `>` ou `NR` ("Not Reported", i.e. não medido).
- **2025:** coluna `<Variável> Status` agora à **direita** da variável, codificada numericamente: `0` = não medido, `1` = medido (não censurado), `2` = abaixo OU acima do LD (direção não informada). As duas primeiras linhas da aba também são lixo (texto descritivo da planilha) e o cabeçalho real fica na terceira linha.

Para manter compatibilidade com o restante do pipeline, todos os anos são unificados na convenção legada:

- Variável não medida → variável `NaN`, LD `NaN`.
- Medida sem censura → LD `NaN`.
- Censurada (`<`, `>` ou `2`) → LD recebe `<` ou `>` (o `2` do 2025 é assumido como `<` por ser o caso dominante em qualidade de água).

## 3. Estruturação do DataFrame Final

O DataFrame final foi estruturado da seguinte forma:

1. Colunas identificadoras:
   - Local (estação de monitoramento)
   - Data (data da coleta)

2. Para cada variável de interesse:
   - Coluna de Limite de Detecção (_LD)
   - Coluna consolidada da variável

## 4. Resultado do Processo de Tratamento

Ao final desta etapa, a base de dados apresenta as seguintes características:

- Ausência de colunas redundantes.
- Variáveis padronizadas e consolidadas.
- Correspondência explícita entre variável e seu respectivo LD.
- Estrutura adequada para análises estatísticas e ambientais subsequentes.

## Utils

### Importações

In [ ]:
import re
import unicodedata
import pandas as pd
from pathlib import Path

### Configurações

In [ ]:
# === Configuração das abas de informações das estações ===
# A partir de 2025, o INEA passou a publicar uma aba separada com layout diferente.
STATIONS_SHEETS = {
    "old": {
        "sheet": "Estações 2012 a 2024",
        "header": 0,
        "code_col": "CÓDIGO ESTAÇÃO",
        "short_code_col": "CÓDIGO ESTAÇÃO RESUMIDO",
        "waterbody_col": "CORPO D'ÁGUA",
    },
    "new": {
        "sheet": "Estações 2025",
        "header": 2,
        "code_col": "Código da estação de monitoramento QA",
        "short_code_col": "Código da estação de monitoramento resumido QA",
        "waterbody_col": "Nome do corpo d'água",
    },
}

# === Configuração por ano de dados ===
# - `header`: linha do cabeçalho (0 para 2012–2024; 2 para 2025 — as duas primeiras linhas são lixo).
# - `stations_key`: qual entrada de STATIONS_SHEETS usar para enriquecer com Codigo Local / Local.
# - `local_col` / `data_col`: nome da coluna que contém o código longo da estação e a data, respectivamente.
# - `status_position`: posição da coluna de LD/Status em relação à variável ("before" para 2012–2024, "after" para 2025).
# - `status_convention`: "lt_gt" (símbolos <, >, NR) ou "012" (códigos numéricos 0, 1, 2).
LEGACY_YEARS = ["2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]

YEAR_CONFIG = {
    **{
        y: {
            "header": 0,
            "stations_key": "old",
            "local_col": "Local",
            "data_col": "Data",
            "status_position": "before",
            "status_convention": "lt_gt",
        }
        for y in LEGACY_YEARS
    },
    "2024": {
        "header": 0,
        "stations_key": "old",
        "local_col": "Local",
        "data_col": "Data",
        "status_position": "before",
        "status_convention": "lt_gt",
    },
    "2025": {
        "header": 2,
        "stations_key": "new",
        "local_col": "Código da estação de monitoramento QA",
        "data_col": "Data da coleta (dd/mm/aaaa)",
        "status_position": "after",
        "status_convention": "012",
    },
}

AVALIABLE_YEARS = list(YEAR_CONFIG.keys())

# === Colunas básicas de identificação ===
WATER_INFORMATION_BASE_COLUMNS = ["Local", "Codigo Local", "Data"]

# === Variáveis de interesse ===
# Cada variável mapeia para a lista de nomes canônicos (após canonical_name) que devem ser tratados
# como equivalentes. Isso permite reconciliar diferenças de nomenclatura entre anos (ex.: "Nitrato" vs
# "Nitratos" em 2025, "Temperatura da Água" vs "TempAmostra" em 2025).
#
# Cianobactérias × Microcistinas é um caso especial: as planilhas trazem a mesma coluna canônica
# "cianotoxinas / microcistinas" para ambas; a distinção vem da unidade (cél/L vs µg/L), que
# canonical_name remove. Por isso essas duas variáveis também declaram um RAW_UNIT_FILTERS (abaixo).
WATER_INFORMATION_INTEREST_COLUMNS = {
    "DBO": ["dbo"],
    "OD": ["od"],
    "Nitrato": ["nitrato", "nitratos"],
    "Nitrogênio Amoniacal Total": ["nitrogenio amoniacal total", "nitrogenio amoniacal"],
    "Fósforo Total": ["fosforo total", "fosforototal"],
    "Condutividade": ["condutividade", "condutividades"],
    "pH": ["ph"],
    "Turbidez": ["turbidez"],
    "Temperatura da Água": ["temperatura da agua", "tempamostra", "temp amostra"],
    "Sólidos Suspensos Totais": ["solidos suspensos totais"],
    "Coliformes Termotolerantes": ["coliformes termotolerantes"],
    "Cianobacterias": ["cianotoxinas / microcistinas", "cianotoxinas/microcistinas"],
    "Microcistinas": ["cianotoxinas / microcistinas", "cianotoxinas/microcistinas"],
}

# Filtros adicionais baseados no nome bruto da coluna (antes de canonical_name), usados para variáveis
# cuja distinção depende da unidade. Cada filtro recebe o nome bruto e retorna True/False.
# - Cianobactérias = contagem de células → unidade "cél/L" (também aceita "cel/L").
# - Microcistinas = concentração       → unidade "µg/L" (também aceita "ug/L" e o caso de 2023/2024
#   sem unidade explícita, no qual os valores observados são compatíveis com µg/L).
def _ciano_is_cells(raw: str) -> bool:
    r = raw.lower()
    return ("cél/l" in r) or ("cel/l" in r) or ("céis/l" in r) or ("celulas" in r)


def _ciano_is_micro(raw: str) -> bool:
    r = raw.lower()
    if "µg/l" in r or "ug/l" in r:
        return True
    # Sem unidade explícita (2023/2024) → Microcistinas (faixa de valores observada é compatível com µg/L)
    has_parens = re.search(r"\(.*?\)", raw) is not None
    return not has_parens


RAW_UNIT_FILTERS = {
    "Cianobacterias": _ciano_is_cells,
    "Microcistinas": _ciano_is_micro,
}

CODIGO_ESTACOES_RESUMIDO = ["JC341", "JC342", "CM320", "MR361", "MR363", "MR369", "TJ303", "TJ306"]  # Códigos resumidos das estações de interesse
CODIGO_ESTACOES = ["01RJ20CM0320", "01RJ20JC0341", "01RJ20JC0342", "01RJ20MR0361", "01RJ20MR0363", "01RJ20MR0369", "01RJ20TJ0303", "01RJ20TJ0306"]  # Códigos longos das estações de interesse

DATA_PATH = Path("../../Data/RawData/WaterQualityRawData.xlsx")  # Caminho para o arquivo de dados brutos

### Funções auxiliares

In [ ]:
# Função para padronizar os nomes das colunas, removendo acentos, convertendo para minúsculas e removendo parênteses e espaços extras
def canonical_name(col):
    col = unicodedata.normalize('NFKD', str(col)).encode('ASCII', 'ignore').decode('ASCII')
    col = col.lower()
    col = re.sub(r'\(.*?\)', '', col)        # remove unidades entre parênteses
    col = re.sub(r'\s+', ' ', col).strip()    # colapsa espaços
    return col


def is_status_col(col_name: str) -> bool:
    """True se o nome da coluna indica um campo de LD/Status (legado ou novo)."""
    norm = canonical_name(col_name)
    # 2012–2023 → "ld" ou "ld.N" (após pd.read_excel renomear duplicatas).
    # 2024–2025 → termina com "status" (ex.: "ph status", " status.1").
    if re.fullmatch(r"ld(\.\d+)?", norm):
        return True
    if norm.endswith("status") or norm == "status":
        return True
    return False


def normalize_status(series: pd.Series, convention: str) -> pd.Series:
    """Converte uma coluna de LD/Status para a convenção unificada legada.

    Saída: string com '<', '>' ou pd.NA.
    - convention='lt_gt' (2012–2024): '<' e '>' permanecem; 'NR'/em branco/qualquer outro vira NA.
    - convention='012'   (2025):       0/1 → NA; 2 → '<' (assumindo censura à esquerda, caso dominante).
    """
    if convention == "lt_gt":
        s = series.astype("string").str.strip().str.upper()
        s = s.where(s.isin(["<", ">"]), pd.NA)
        return s
    if convention == "012":
        s = pd.to_numeric(series, errors="coerce")
        return s.map({0: pd.NA, 1: pd.NA, 2: "<"}).astype("string")
    raise ValueError(f"Convenção de status desconhecida: {convention!r}")


def find_adjacent_status_col(columns: list[str], var_idx: int, position: str) -> str | None:
    """Retorna o nome da coluna de LD/Status adjacente ao índice da variável, ou None."""
    target_idx = var_idx - 1 if position == "before" else var_idx + 1
    if 0 <= target_idx < len(columns) and is_status_col(columns[target_idx]):
        return columns[target_idx]
    return None

## Código

### Aquisição e Consolidação Inicial dos Dados

Os dados foram obtidos a partir de planilhas anuais disponibilizadas pelo INEA.  

As etapas iniciais consistiram em:

1. Leitura de todos os arquivos anuais disponíveis.
2. Consolidação dos dados em um único DataFrame.
3. Filtragem dos registros com base nos códigos das estações de monitoramento de interesse.

Essa etapa garantiu que apenas os pontos amostrais relevantes para o estudo fossem considerados nas análises subsequentes.

In [ ]:
# Lê e unifica as duas abas de informações das estações em um único DataFrame
def load_stations() -> pd.DataFrame:
    frames = []
    for cfg in STATIONS_SHEETS.values():
        df = pd.read_excel(DATA_PATH, sheet_name=cfg["sheet"], header=cfg["header"])
        df = df[df[cfg["short_code_col"]].isin(CODIGO_ESTACOES_RESUMIDO)]
        df = df[[cfg["short_code_col"], cfg["code_col"], cfg["waterbody_col"]]].rename(
            columns={
                cfg["short_code_col"]: "Codigo Local",
                cfg["code_col"]: "CodigoEstacao",
                cfg["waterbody_col"]: "Local",
            }
        )
        frames.append(df)
    # Algumas estações aparecem nas duas abas; mantemos a primeira ocorrência (preferimos a aba legada).
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset="CodigoEstacao")


df_estacoes = load_stations()
df_estacoes

In [ ]:
# Carrega cada ano aplicando sua configuração específica e produz um DataFrame por ano
# já no formato final: Local/Data + para cada variável de interesse, colunas <var> e <var>_LD.

def load_year(year: str) -> pd.DataFrame:
    cfg = YEAR_CONFIG[year]
    df = pd.read_excel(DATA_PATH, sheet_name=year, header=cfg["header"])

    # Filtra estações de interesse
    df = df[df[cfg["local_col"]].isin(CODIGO_ESTACOES)].copy()
    if df.empty:
        return pd.DataFrame()

    cols = list(df.columns)
    norm = {c: canonical_name(c) for c in cols}

    out = pd.DataFrame(index=df.index)
    out["CodigoEstacao"] = df[cfg["local_col"]].values
    out["Data"] = pd.to_datetime(df[cfg["data_col"]], errors="coerce").values

    for var, synonyms in WATER_INFORMATION_INTEREST_COLUMNS.items():
        synonyms_set = set(synonyms)
        # Encontra colunas cujo nome canônico bate com algum sinônimo
        var_cols = [c for c in cols if norm[c] in synonyms_set]
        # Filtro adicional para variáveis cuja distinção depende da unidade (Cianobactérias × Microcistinas)
        raw_filter = RAW_UNIT_FILTERS.get(var)
        if raw_filter is not None:
            var_cols = [c for c in var_cols if raw_filter(c)]
        if not var_cols:
            continue

        val_frames = []
        ld_frames = []
        for vc in var_cols:
            v_idx = cols.index(vc)
            values = pd.to_numeric(df[vc], errors="coerce")

            ld_col = find_adjacent_status_col(cols, v_idx, cfg["status_position"])
            if ld_col is not None:
                ld_series = normalize_status(df[ld_col], cfg["status_convention"])
                # Em 2025 (convenção 012), status==0 significa "não medido", mas a planilha pode trazer
                # 0.0 placeholder na coluna da variável. Nessas linhas, zeramos o valor para NaN.
                if cfg["status_convention"] == "012":
                    not_measured = pd.to_numeric(df[ld_col], errors="coerce") == 0
                    values = values.mask(not_measured.values)
                ld_frames.append(ld_series.reset_index(drop=True))
            val_frames.append(values.reset_index(drop=True))

        # Consolida múltiplas colunas redundantes pela média linha-a-linha
        out[var] = pd.concat(val_frames, axis=1).mean(axis=1).values

        if ld_frames:
            ld_df = pd.concat(ld_frames, axis=1)
            # Coalesce: primeira marca não-nula encontrada entre as colunas redundantes
            out[f"{var}_LD"] = ld_df.bfill(axis=1).iloc[:, 0].values

    out["Ano"] = year
    return out


df_water_data = pd.concat([load_year(y) for y in AVALIABLE_YEARS], ignore_index=True)

# Enriquece com Codigo Local (resumido) e Local (corpo d'água)
df_water_data = df_water_data.merge(
    df_estacoes[["CodigoEstacao", "Codigo Local", "Local"]],
    on="CodigoEstacao",
    how="left",
).drop(columns=["CodigoEstacao"])

# Reordena: base columns primeiro, depois pares (LD, valor) por variável
ordered = ["Local", "Codigo Local", "Data", "Ano"]
for var in WATER_INFORMATION_INTEREST_COLUMNS:
    if f"{var}_LD" in df_water_data.columns:
        ordered.append(f"{var}_LD")
    if var in df_water_data.columns:
        ordered.append(var)
df_water_data = df_water_data[[c for c in ordered if c in df_water_data.columns]]

display(df_water_data)

### Remoção de Colunas Integralmente Vazias

Foi realizada a identificação e remoção de:

- Colunas cujos registros estavam integralmente vazios.
- Suas respectivas colunas associadas de Limite de Detecção (LD).

A exclusão dessas colunas reduz ruído estrutural e melhora a eficiência computacional das etapas posteriores.

In [ ]:
# Como o loader por ano já seleciona apenas as variáveis de interesse, não restam colunas "residuais" para descartar.
# Apenas reportamos quantos registros e variáveis há na base consolidada.
print(f"Registros: {len(df_water_data):,}")
print(f"Variáveis numéricas: {sum(c in df_water_data.columns for c in WATER_INFORMATION_INTEREST_COLUMNS)}")
print("Registros por ano:")
print(df_water_data['Ano'].value_counts().sort_index())

### Identificação de Inconsistências Estruturais

Durante a inspeção exploratória da base de dados, foram identificadas inconsistências estruturais, incluindo:

- Colunas duplicadas representando a mesma variável.
- Diferenças de nomenclatura decorrentes de:
  - Espaços adicionais.
  - Diferenças entre letras maiúsculas e minúsculas.
  - Presença ou ausência de unidades no nome da coluna.
  - Variações na acentuação.

Exemplos observados:

- "pH" e "pH  "
- "OD (mg/L)" e "OD  (mg/L)"
- "Nitrogênio amoniacal total (mg/L)" e "Nitrogênio Amoniacal Total (mg/L)"

Essas inconsistências poderiam comprometer análises automatizadas e gerar duplicidade indevida de variáveis.

### Padronização de Nomenclatura

Foi implementado um processo sistemático de padronização para:

- Remover acentuação.
- Eliminar unidades descritas entre parênteses.
- Normalizar espaços múltiplos.
- Uniformizar caixa de texto (lowercase).

Essa abordagem permitiu identificar colunas semanticamente equivalentes, independentemente de variações superficiais de escrita.

### Consolidação de Variáveis Duplicadas

Após a padronização, variáveis equivalentes foram consolidadas em colunas únicas.

A regra aplicada foi:

- Se apenas uma coluna possuía valor no registro → esse valor foi mantido.
- Se múltiplas colunas possuíam valores no mesmo registro → foi calculada a média aritmética linha a linha.
- Se todas estavam ausentes → o valor permaneceu como ausente.

Essa estratégia assegura:

- Coerência estatística.
- Preservação da informação disponível.
- Eliminação de redundâncias estruturais.

In [ ]:
# Cianobactérias × Microcistinas: a desambiguação é feita dentro de load_year() via RAW_UNIT_FILTERS.
# Esta célula é mantida apenas como ponto de auditoria do que foi resolvido.
ciano_summary = df_water_data[[c for c in df_water_data.columns if c.startswith(("Cianobacterias", "Microcistinas"))]].notna().sum()
print("Contagem de valores não-nulos em Cianobactérias / Microcistinas:")
print(ciano_summary)

In [ ]:
# A consolidação por nome canônico e o emparelhamento variável↔LD agora são feitos dentro de load_year().
# Aqui apenas materializamos o resultado final.
df_final = df_water_data.copy()

In [ ]:
# Sanidade: garante que para toda variável presente exista a coluna *_LD correspondente.
faltando_ld = [
    var
    for var in WATER_INFORMATION_INTEREST_COLUMNS
    if var in df_final.columns and f"{var}_LD" not in df_final.columns
]
if faltando_ld:
    print("Aviso: variáveis sem coluna de LD associada:", faltando_ld)
else:
    print("Todas as variáveis possuem coluna de LD associada.")

In [ ]:
df_final.head()

In [ ]:
df_final.to_excel("../../Data/IntermediaryData/WaterQualityInitialData.xlsx", index=False)